In [1]:
from utils import *
import wandb
from datetime import datetime

In [2]:
def itertoolsBetter(dataIter):
    while True:
        for batch in dataIter:
            yield batch

In [3]:
# https://github.com/kreasof-ai/sigreg
def sigreg_weak_loss(x, sketch_dim=64):
    """
    Forces Covariance(x) ~ Identity.
    Matches the 2nd Moment (Spherical Cloud).
    """
    N, C = x.size()
    # 1. Sketching (Optional for C=512, but good for consistency)
    if C > sketch_dim:
        S = torch.randn(sketch_dim, C, device=x.device) / (C ** 0.5)
        x = x @ S.T  # [N, sketch_dim]
    else:
        sketch_dim = C

    # 2. Centering & Covariance
    x = x - x.mean(dim=0, keepdim=True)
    cov = (x.T @ x) / (N - 1 + 1e-6)

    # 3. Target Identity
    target = torch.eye(sketch_dim, device=x.device)

    # 4. Off-diagonal suppression + Diagonal maintenance
    return torch.norm(cov - target, p='fro')


class EmbeddingLoss(nn.Module):
    def __init__(self, temperature=0.04, alpha=0.1):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha

    def forward(self, x, y, families=None):
        B = x.shape[0]
        xNorm, yNorm = nn.functional.normalize(x, dim=-1), nn.functional.normalize(y, dim=-1)

        logits = (xNorm @ yNorm.T) / self.temperature  # [B, B]

        if families is not None:
            # True where i and j share a family — these are positives
            familyMatch = (families.unsqueeze(0) == families.unsqueeze(1))  # [B, B]
            # Diagonal is always a positive
            familyMatch.fill_diagonal_(True)

            # Soft target: uniform distribution over all positives in the row
            targets = familyMatch.float()
            targets = targets / targets.sum(dim=1, keepdim=True)  # normalize rows to sum to 1

            # Mask diagonal out of the denominator for the standard InfoNCE denominator
            # but keep all positives in the numerator
            logSoftmax = torch.log_softmax(logits, dim=1)
            loss1 = -(targets * logSoftmax).sum(dim=1).mean()

            logSoftmax2 = torch.log_softmax(logits.T, dim=1)
            loss2 = -(targets.T * logSoftmax2).sum(dim=1).mean()
        else:
            # Fall back to standard InfoNCE with integer labels
            labels = torch.arange(B, device=x.device)
            loss1 = nn.functional.cross_entropy(logits, labels)
            loss2 = nn.functional.cross_entropy(logits.T, labels)

        infoLoss = 0.5 * (loss1 + loss2)
        sigLoss = 0.5 * (sigreg_weak_loss(x) + sigreg_weak_loss(y))

        return {"total": infoLoss + self.alpha * sigLoss, "info": infoLoss, "sig": sigLoss}


class Perplexity(nn.Module):
    def __init__(self, loss):
        super().__init__()
        self.loss = loss

    def forward(self, y1, y2, names):
        log = self.loss(y1, y2, names)["info"]
        return torch.exp(log)

In [4]:
def saveExperiment(imageModel, textModel, config, experimentName, start):
    latestPath = os.path.join("checkpoints", "finetune", "latest")
    if not os.path.exists(os.path.join("checkpoints", "finetune", "latest")):
        os.mkdir(latestPath)

    stamp = start.strftime("%Y-%m-%d %H-%M")
    timePath = os.path.join("checkpoints", "finetune", stamp)
    if not os.path.exists(timePath):
        os.mkdir(timePath)

    saveToPath(latestPath, imageModel, textModel, config, experimentName)
    saveToPath(timePath, imageModel, textModel, config, experimentName)


def saveToPath(path, imageModel, textModel, config, experimentName):
    if not os.path.exists(os.path.join(path, experimentName)):
        os.mkdir(os.path.join(path, experimentName))

    torch.save(imageModel.state_dict(), os.path.join(path, experimentName, "image.pt"))
    textModel.save_pretrained(os.path.join(path, experimentName, "text"))
    # torch.save(textModel.state_dict(), os.path.join(path, experimentName, "text.pt"))
    config.save(os.path.join(path, experimentName, "config.json"))

In [5]:
def trainModel(config, textModel, imageModel, dataset, experimentName, start, imageConfig):
    optimizer = torch.optim.Adam(list(imageModel.head.parameters()) + list(textModel.head.parameters()), lr=config.learningRate)

    imageModel.to(device)
    textModel.to(device)

    objective = EmbeddingLoss(temperature=0.04, alpha=0.1)
    # objective = ContrastiveLoss2()
    criterion = Perplexity(objective)

    train, test = dataset.split(dataset, batchSize=config.batchSize)

    testIter = itertoolsBetter(test)

    testHistory = []

    total = 0

    run = None

    try:
        for epoch in range(config.epochs):
            progress = 0
            averageTrainLoss = 0
            averageTestLoss = 0
            for images, info, text, _ in train:
                imageModel.train()
                textModel.train()
                optimizer.zero_grad()

                imageOutputs = imageModel(images.to(device))
                textOutputs = textModel(text.to(device))
                loss = objective(imageOutputs, textOutputs, info.to(device))
                trainPerplexity = criterion(imageOutputs, textOutputs, info.to(device))

                trainLoss = loss["total"].detach().item()
                averageTrainLoss = (averageTrainLoss * progress + trainLoss) / (progress + 1)

                loss["total"].backward()
                optimizer.step()

                with torch.no_grad():
                    imageModel.eval()
                    textModel.eval()
                    images1, info1, text1, _ = next(testIter)
                    imageOutputs1 = imageModel(images1.to(device))
                    textOutputs1 = textModel(text1.to(device))
                    loss1 = objective(imageOutputs1, textOutputs1, info1.to(device))
                    testPerplexity = criterion(imageOutputs1, textOutputs1, info1.to(device))

                    testLoss = loss1["total"].detach().item()
                    averageTestLoss = (averageTestLoss * progress + testLoss) / (progress + 1)

                if run is None:
                    run = wandb.init(entity="dylanberndt123-missouri-state-university", project="Briefcase", config=config.serialize())
                
                run.log({"Train Loss": trainLoss, 
                         "Train Perplexiy": trainPerplexity.detach().item(),
                         "Train InfoNCE": loss["info"].detach().item(),
                         "Train SigREG": loss["sig"].detach().item(),
                         "Test Loss": testLoss,
                         "Test Perplexity": testPerplexity.detach().item(),
                         "Test InfoNCE": loss1["info"].detach().item(),
                         "Test SigREG": loss1["sig"].detach().item()
                         }, step=total)

                progress += 1
                total += 1

                progressString = f"\r{epoch + 1} | {progress}/{len(train)} | {(progress / len(train)) * 100:.3f}%"

                print(f"{progressString} |  Train Loss: {averageTrainLoss:.2f} | Test Loss: {averageTestLoss:.2f}",end="")

            print(f"\rEpoch {epoch + 1} | Train Loss: {averageTrainLoss:.2f} | Test Loss: {averageTestLoss:.2f}{' ' * 50}")

            if (np.array(testHistory) < averageTestLoss).sum() >= 2:
                raise KeyboardInterrupt
            testHistory.append(averageTestLoss)

    except KeyboardInterrupt:
        saveExperiment(imageModel, textModel, imageConfig, experimentName, start)
        return imageModel, textModel

    saveExperiment(imageModel, textModel, imageConfig, experimentName, start)
    return imageModel, textModel

In [6]:
queryConfig = Config().load(os.path.join("configs", "querying.json"))

In [ ]:
sharedDim = queryConfig.sharedDim

# textModelName = "openai/clip-vit-base-patch32"
# textModel = CLIPTextEmbedder(textModelName, sharedDim).to(device)

textModelName = "bert-base-uncased"
textModel = BERTTextEmbedder(textModelName, sharedDim).to(device)

vit, imageConfig = ViT.load(os.path.join("checkpoints", "pretrain", "latest"))
imageModel = ViTEmbedder(vit, sharedDim).to(device)

dataset = CombinedQueryData(imageConfig.dataset)
dataset.method = "none"

dataset.setTokenizer(textModelName)

experimentName = "ViT" + " " + textModelName.replace("/", "-")

imageModel, textModel = trainModel(queryConfig, textModel, imageModel, dataset, experimentName, datetime.now(), imageConfig)

saveExperiment(imageModel, textModel, imageConfig, experimentName, datetime.now())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Loading MyFonts images from dataset ====================


Fonts serialized: 720/3870google/fonts/ofl/kumarone/KumarOne-Regular.ttf execution context too long
Fonts serialized: 939/3870google/fonts/ofl/kumaroneoutline/KumarOneOutline-Regular.ttf execution context too long
Fonts serialized: 2449/3870google/fonts/ofl/notocoloremojicompattest/NotoColorEmojiCompatTest-Regular.ttf invalid pixel size
Fonts serialized: 3870/3870


Fonts serialized: 1395/18624dafont/fonts/citaro_voor_dubbele_hoogte_middendubbel.ttf [Errno 2] No such file or directory: 'dafont/bitmaps/Citaro Voor (dubbele hoogte, midden/dubbel) Regular al.bmp'
Fonts serialized: 3251/18624dafont/fonts/mischess.ttf corrupt cmap table format 4 (data length: 642, header length: 2136)
Fonts serialized: 3469/18624dafont/fonts/wi5med_grid.ttf [Errno 2] No such file or directory: 'dafont/bitmaps/wi/5Med Grid Regular al.bmp'
Fonts serialized: 5344/18624dafont/fonts/kaylafiz.ttf division by zero
Fonts serialized: 5481/18624dafont/fonts/

cmap subtable is reported as having zero length: platformID 1, platEncID 0, format 0 offset 20. Skipping table.
cmap subtable is reported as having zero length: platformID 1, platEncID 0, format 0 offset 20. Skipping table.


Fonts serialized: 7691/18624dafont/fonts/assq.ttf [Errno 2] No such file or directory: 'dafont/bitmaps/AS/SQ Regular al.bmp'
Fonts serialized: 8112/18624dafont/fonts/50fifty.ttf [Errno 2] No such file or directory: 'dafont/bitmaps/50/fifty Regular al.bmp'
Fonts serialized: 8659/18624dafont/fonts/deborah_extras-ornaments.ttf [Errno 2] No such file or directory: 'dafont/bitmaps/Deborah Extras/Ornaments Regular al.bmp'
Fonts serialized: 9030/18624dafont/fonts/slicedmeats.ttf [Errno 2] No such file or directory: 'dafont/bitmaps/Vladimir Nikolic https://www.coroflot.com/vladimirnikolic al.bmp'
Fonts serialized: 9876/18624

...

Fonts serialized: 10370/18624dafont/fonts/humbug.ttf corrupt cmap table format 0 (data length: 534, header length: 598)
Fonts serialized: 10579/18624dafont/fonts/no-regular-inline.ttf division by zero
Fonts serialized: 11433/18624dafont/fonts/zfraktur-eye-fs.ttf [Errno 2] No such file or directory: 'dafont/bitmaps/zfraktur eYe/FS Regular al.bmp'
Fonts serialized: 11496/18624dafont/fonts/julia_black_extended.ttf Not a TrueType or OpenType font (bad sfntVersion)
Fonts serialized: 13185/18624dafont/fonts/ztorm_eyefs.ttf [Errno 2] No such file or directory: 'dafont/bitmaps/ztorm eYe/FS Regular al.bmp'
Fonts serialized: 13434/18624dafont/fonts/subtle.ttf corrupt cmap table format 0 (data length: 526, header length: 598)
Fonts serialized: 14122/18624dafont/fonts/zoulsister_plus_eye_fs.ttf [Errno 2] No such file or directory: 'dafont/bitmaps/zoulsister plus eYe/FS Regular al.bmp'
Fonts serialized: 15687/18624dafont/fonts/wc_wunderbach_rough.otf cannot open resource
Fonts serialized: 16321/186

/home/ubuntu/Briefcase/utils/loaders/dafont.py:14: ParserWarning: Skipping line 2205: expected 6 fields, saw 8
Skipping line 3214: expected 6 fields, saw 7
Skipping line 7537: expected 6 fields, saw 7
Skipping line 7538: expected 6 fields, saw 7
Skipping line 7539: expected 6 fields, saw 7
Skipping line 7931: expected 6 fields, saw 8
Skipping line 7932: expected 6 fields, saw 8
Skipping line 8343: expected 6 fields, saw 8
Skipping line 9171: expected 6 fields, saw 7
Skipping line 9419: expected 6 fields, saw 8
Skipping line 9420: expected 6 fields, saw 7
Skipping line 9470: expected 6 fields, saw 8
Skipping line 9856: expected 6 fields, saw 8
Skipping line 9934: expected 6 fields, saw 7
Skipping line 10059: expected 6 fields, saw 7
Skipping line 10060: expected 6 fields, saw 7
Skipping line 10092: expected 6 fields, saw 8
Skipping line 10093: expected 6 fields, saw 8
Skipping line 10096: expected 6 fields, saw 7
Skipping line 10134: expected 6 fields, saw 7
Skipping line 11547: expecte

43.23% of fonts have descriptions


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ubuntu/.netrc.
wandb: Currently logged in as: dylanberndt123 (dylanberndt123-missouri-state-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1 | Train Loss: 7.18 | Test Loss: 7.05                                                  
2 | 724/767 | 94.394% |  Train Loss: 6.95 | Test Loss: 6.81